In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import gc
import json
import random
import shutil
import hashlib
from pathlib import Path

import numpy as np
import tensorflow as tf


SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


print("TensorFlow:", tf.__version__)
print("Seed:", SEED)

gpus = tf.config.list_physical_devices("GPU")
print("GPU(s):", gpus)

TensorFlow: 2.20.0
Seed: 42
GPU(s): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/CAMUS_FINAL"
)

PHASE1_DIR = (
    PROJECT_DIR /
    "phase1_segmentation"
)

P1D_DIR = (
    PHASE1_DIR /
    "phase1D_loss"
)

P1E_DIR = (
    PHASE1_DIR /
    "phase1E_final_validation"
)

P1F_DIR = (
    PHASE1_DIR /
    "phase1F_final_model"
)

P1F_MODEL_DIR = (
    P1F_DIR /
    "models"
)

P1F_CONFIG_DIR = (
    P1F_DIR /
    "configs"
)

P1F_TABLE_DIR = (
    P1F_DIR /
    "tables"
)


for folder in [
    P1F_MODEL_DIR,
    P1F_CONFIG_DIR,
    P1F_TABLE_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


print("Phase 1F directory:")
print(P1F_DIR)

print("\n✓ Final-model folders ready.")

Phase 1F directory:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1F_final_model

✓ Final-model folders ready.


In [ ]:
P1D_DECISION_PATH = (
    P1D_DIR /
    "configs" /
    "phase1D_loss_decision.json"
)

P1E_RECORD_PATH = (
    P1E_DIR /
    "configs" /
    "phase1E_final_validation_record.json"
)


assert P1D_DECISION_PATH.exists(), (
    f"Missing Phase 1D decision:\n"
    f"{P1D_DECISION_PATH}"
)

assert P1E_RECORD_PATH.exists(), (
    f"Missing Phase 1E record:\n"
    f"{P1E_RECORD_PATH}"
)


with open(
    P1D_DECISION_PATH,
    "r"
) as f:
    phase1d_decision = json.load(f)


with open(
    P1E_RECORD_PATH,
    "r"
) as f:
    phase1e_record = json.load(f)


SOURCE_CHECKPOINT = Path(
    phase1d_decision[
        "selected_checkpoint"
    ]
)


assert SOURCE_CHECKPOINT.exists()

assert (
    phase1d_decision[
        "selected_loss_key"
    ]
    == "bce_dice"
)

assert (
    phase1e_record[
        "status"
    ]
    == "COMPLETED"
)

assert (
    phase1e_record[
        "test_driven_tuning_performed"
    ]
    is False
)


print("=" * 70)
print("PHASE 1D + PHASE 1E RECORDS VERIFIED")
print("=" * 70)

print("\nAccepted source checkpoint:")
print(SOURCE_CHECKPOINT)

print(
    "\nPhase 1E status:",
    phase1e_record["status"]
)

print(
    "Post-test tuning:",
    phase1e_record[
        "test_driven_tuning_performed"
    ]
)

print(
    "\n✓ Accepted model provenance verified."
)

PHASE 1D + PHASE 1E RECORDS VERIFIED

Accepted source checkpoint:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/models/bce_dice_best.keras

Phase 1E status: COMPLETED
Post-test tuning: False

✓ Accepted model provenance verified.


In [ ]:
final_config = (
    phase1e_record[
        "final_configuration"
    ]
)


assert (
    final_config["resolution"]
    == 256
)

assert (
    final_config["architecture"]
    == "Large U-Net"
)

assert (
    final_config["filters"]
    == [32, 64, 128, 256, 512]
)

assert (
    final_config["parameters"]
    == 7_759_521
)

assert (
    final_config["optimizer"]
    == "Adam"
)

assert np.isclose(
    final_config[
        "learning_rate"
    ],
    1e-4
)

assert (
    final_config["batch_size"]
    == 2
)

assert (
    final_config["augmentation"]
    is False
)

assert (
    final_config["loss"]
    == "BCE + Dice"
)

assert np.isclose(
    final_config[
        "threshold"
    ],
    0.50
)


print("=" * 70)
print("ACCEPTED FINAL CONFIGURATION")
print("=" * 70)

print("\nResolution    : 256×256")
print("Architecture  : Large U-Net")
print("Filters       : [32, 64, 128, 256, 512]")
print("Parameters    : 7,759,521")
print("Optimizer     : Adam")
print("Learning rate : 1e-4")
print("Batch size    : 2")
print("Augmentation  : None")
print("Loss          : BCE + Dice")
print("Threshold     : 0.50")
print("Postprocess   : LCC + hole filling")

print(
    "\n✓ Configuration verified."
)

ACCEPTED FINAL CONFIGURATION

Resolution    : 256×256
Architecture  : Large U-Net
Filters       : [32, 64, 128, 256, 512]
Parameters    : 7,759,521
Optimizer     : Adam
Learning rate : 1e-4
Batch size    : 2
Augmentation  : None
Loss          : BCE + Dice
Threshold     : 0.50
Postprocess   : LCC + hole filling

✓ Configuration verified.


In [ ]:
def sha256_file(file_path):

    sha = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            sha.update(chunk)

    return sha.hexdigest()


source_sha256 = sha256_file(
    SOURCE_CHECKPOINT
)


source_size_bytes = (
    SOURCE_CHECKPOINT
    .stat()
    .st_size
)


print("=" * 70)
print("SOURCE MODEL FINGERPRINT")
print("=" * 70)

print(
    "\nFile:",
    SOURCE_CHECKPOINT
)

print(
    "\nSize:",
    f"{source_size_bytes:,}",
    "bytes"
)

print(
    "\nSHA-256:"
)

print(source_sha256)

SOURCE MODEL FINGERPRINT

File: /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/models/bce_dice_best.keras

Size: 93,271,247 bytes

SHA-256:
65ec53959831c6b7c08747a5d94de6563580ce9ec052aa7b9afa614bee5705d3


In [ ]:
FINAL_MODEL_PATH = (
    P1F_MODEL_DIR /
    "final_lv_segmentation_model.keras"
)


if FINAL_MODEL_PATH.exists():

    print(
        "Existing final model found."
    )

    existing_hash = sha256_file(
        FINAL_MODEL_PATH
    )

    if (
        existing_hash
        != source_sha256
    ):

        raise RuntimeError(
            "A different model already exists at "
            "the final-model path. It will NOT be "
            "overwritten automatically."
        )

    print(
        "✓ Existing final model is "
        "already identical to source."
    )

else:

    shutil.copy2(
        SOURCE_CHECKPOINT,
        FINAL_MODEL_PATH
    )

    print(
        "✓ Accepted checkpoint copied "
        "to final-model directory."
    )


print("\nFinal model:")
print(FINAL_MODEL_PATH)

✓ Accepted checkpoint copied to final-model directory.

Final model:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1F_final_model/models/final_lv_segmentation_model.keras


In [ ]:
final_sha256 = sha256_file(
    FINAL_MODEL_PATH
)

final_size_bytes = (
    FINAL_MODEL_PATH
    .stat()
    .st_size
)


assert (
    final_sha256
    ==
    source_sha256
)

assert (
    final_size_bytes
    ==
    source_size_bytes
)


print("=" * 70)
print("BYTE-LEVEL VERIFICATION")
print("=" * 70)

print(
    "\nSource SHA-256:"
)

print(source_sha256)

print(
    "\nFinal SHA-256:"
)

print(final_sha256)

print(
    "\nIdentical:",
    final_sha256 == source_sha256
)

print(
    "\n✓ Final model is a byte-for-byte "
    "copy of the accepted checkpoint."
)

BYTE-LEVEL VERIFICATION

Source SHA-256:
65ec53959831c6b7c08747a5d94de6563580ce9ec052aa7b9afa614bee5705d3

Final SHA-256:
65ec53959831c6b7c08747a5d94de6563580ce9ec052aa7b9afa614bee5705d3

Identical: True

✓ Final model is a byte-for-byte copy of the accepted checkpoint.


In [ ]:
SMOOTH = 1e-6


def dice_coefficient(
    y_true,
    y_pred
):

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred,
        tf.float32
    )

    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3]
    )

    denominator = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3]
        )
    )

    dice = (
        2.0 * intersection
        + SMOOTH
    ) / (
        denominator
        + SMOOTH
    )

    return tf.reduce_mean(
        dice
    )


def iou_metric(
    y_true,
    y_pred
):

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred,
        tf.float32
    )

    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3]
    )

    union = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3]
        )
        -
        intersection
    )

    iou = (
        intersection
        + SMOOTH
    ) / (
        union
        + SMOOTH
    )

    return tf.reduce_mean(
        iou
    )


def dice_loss(
    y_true,
    y_pred
):

    return (
        1.0 -
        dice_coefficient(
            y_true,
            y_pred
        )
    )


def bce_dice_loss(
    y_true,
    y_pred
):

    bce = (
        tf.keras.losses
        .binary_crossentropy(
            y_true,
            y_pred
        )
    )

    bce = tf.reduce_mean(
        bce
    )

    return (
        bce +
        dice_loss(
            y_true,
            y_pred
        )
    )


CUSTOM_OBJECTS = {

    "bce_dice_loss":
        bce_dice_loss,

    "dice_loss":
        dice_loss,

    "dice_coefficient":
        dice_coefficient,

    "iou_metric":
        iou_metric
}


print("✓ Custom objects ready.")

✓ Custom objects ready.


In [ ]:
tf.keras.backend.clear_session()
gc.collect()


source_segmentation_model = (
    tf.keras.models.load_model(
        SOURCE_CHECKPOINT,
        custom_objects=CUSTOM_OBJECTS,
        compile=False
    )
)


frozen_segmentation_model = (
    tf.keras.models.load_model(
        FINAL_MODEL_PATH,
        custom_objects=CUSTOM_OBJECTS,
        compile=False
    )
)


assert (
    source_segmentation_model.count_params()
    ==
    7_759_521
)

assert (
    frozen_segmentation_model.count_params()
    ==
    7_759_521
)

assert (
    source_segmentation_model.input_shape
    ==
    (None, 256, 256, 1)
)

assert (
    frozen_segmentation_model.input_shape
    ==
    (None, 256, 256, 1)
)

assert (
    source_segmentation_model.output_shape
    ==
    (None, 256, 256, 1)
)

assert (
    frozen_segmentation_model.output_shape
    ==
    (None, 256, 256, 1)
)


print("=" * 70)
print("MODEL-LEVEL VERIFICATION")
print("=" * 70)

print(
    "\nParameters:",
    f"{frozen_segmentation_model.count_params():,}"
)

print(
    "Input shape :",
    frozen_segmentation_model.input_shape
)

print(
    "Output shape:",
    frozen_segmentation_model.output_shape
)

print(
    "\n✓ Source and frozen models "
    "load successfully."
)

MODEL-LEVEL VERIFICATION

Parameters: 7,759,521
Input shape : (None, 256, 256, 1)
Output shape: (None, 256, 256, 1)

✓ Source and frozen models load successfully.


In [ ]:
rng = np.random.default_rng(
    SEED
)

verification_input = (
    rng.random(
        (
            1,
            256,
            256,
            1
        ),
        dtype=np.float32
    )
)


source_prediction = (
    source_segmentation_model(
        verification_input,
        training=False
    )
    .numpy()
)


frozen_prediction = (
    frozen_segmentation_model(
        verification_input,
        training=False
    )
    .numpy()
)


max_abs_difference = float(
    np.max(
        np.abs(
            source_prediction
            -
            frozen_prediction
        )
    )
)


mean_abs_difference = float(
    np.mean(
        np.abs(
            source_prediction
            -
            frozen_prediction
        )
    )
)


assert np.array_equal(
    source_prediction,
    frozen_prediction
)


print("=" * 70)
print("PREDICTION EQUIVALENCE CHECK")
print("=" * 70)

print(
    "\nMaximum absolute difference:",
    max_abs_difference
)

print(
    "Mean absolute difference   :",
    mean_abs_difference
)

print(
    "\nExact prediction match:",
    np.array_equal(
        source_prediction,
        frozen_prediction
    )
)

print(
    "\n✓ Frozen model reproduces "
    "the source model exactly."
)

PREDICTION EQUIVALENCE CHECK

Maximum absolute difference: 0.0
Mean absolute difference   : 0.0

Exact prediction match: True

✓ Frozen model reproduces the source model exactly.


In [ ]:
FINAL_CONFIG_PATH = (
    P1F_CONFIG_DIR /
    "final_segmentation_config.json"
)


final_segmentation_config = {

    "model_name":
        "CAMUS A4C LV Final Segmentation Model",

    "task":
        "Left ventricular cavity segmentation",

    "view":
        "A4C",

    "phases":
        ["ED", "ES"],

    "dataset":
        "CAMUS",

    "input": {

        "resolution":
            [256, 256],

        "channels":
            1,

        "normalization":
            "per-image min-max normalization",

        "resize":
            "stretch resize to 256x256"
    },

    "architecture": {

        "name":
            "Large U-Net",

        "filters":
            [32, 64, 128, 256, 512],

        "parameters":
            7_759_521,

        "output_activation":
            "sigmoid"
    },

    "training": {

        "optimizer":
            "Adam",

        "learning_rate":
            1e-4,

        "batch_size":
            2,

        "augmentation":
            False,

        "loss":
            "BCE + Dice",

        "maximum_epochs":
            30,

        "selected_checkpoint_epoch":
            int(
                phase1d_decision[
                    "selected_validation_results"
                ].get(
                    "best_epoch",
                    28
                )
            )
            if isinstance(
                phase1d_decision[
                    "selected_validation_results"
                ],
                dict
            )
            else 28,

        "seed":
            42
    },

    "inference": {

        "threshold":
            0.50,

        "postprocessing": [
            "largest connected component",
            "binary hole filling"
        ]
    },

    "final_model_path":
        str(
            FINAL_MODEL_PATH
        ),

    "sha256":
        final_sha256,

    "phase1e_test_results":
        phase1e_record[
            "final_test_results"
        ],

    "phase2_measurement_note":
        (
            "Quantitative physical measurements must use "
            "the original NIfTI pixel spacing and account "
            "for the geometric scaling introduced by "
            "256x256 stretch resizing."
        ),

    "frozen":
        True
}


with open(
    FINAL_CONFIG_PATH,
    "w"
) as f:

    json.dump(
        final_segmentation_config,
        f,
        indent=2
    )


print("Final configuration saved:")
print(FINAL_CONFIG_PATH)

Final configuration saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1F_final_model/configs/final_segmentation_config.json


In [ ]:
MANIFEST_PATH = (
    P1F_CONFIG_DIR /
    "final_segmentation_manifest.json"
)


final_manifest = {

    "status":
        "FROZEN",

    "project":
        "Computer Vision Based Assistive Application for Analyzing ECHO Scans — LV A4C",

    "phase":
        "Phase 1F — Freeze Final Segmentation Model",

    "source_checkpoint":
        str(
            SOURCE_CHECKPOINT
        ),

    "frozen_model":
        str(
            FINAL_MODEL_PATH
        ),

    "source_sha256":
        source_sha256,

    "frozen_sha256":
        final_sha256,

    "byte_identical":
        bool(
            source_sha256
            ==
            final_sha256
        ),

    "prediction_equivalence_verified":
        True,

    "prediction_max_abs_difference":
        max_abs_difference,

    "selection_history": {

        "Phase_1A":
            (
                "256x256 resolution selected"
            ),

        "Phase_1B1":
            (
                "learning rate 1e-4 selected"
            ),

        "Phase_1B2":
            (
                "Adam retained"
            ),

        "Phase_1B3":
            (
                "batch size 2 selected"
            ),

        "Phase_1B4":
            (
                "no augmentation selected"
            ),

        "Phase_1C":
            (
                "Large U-Net selected"
            ),

        "Phase_1D":
            (
                "BCE + Dice selected"
            ),

        "Phase_1E":
            (
                "locked final segmentation evaluation completed"
            )
    },

    "test_driven_tuning_after_phase1e":
        False,

    "model_is_frozen_for_phase2":
        True,

    "phase2_instruction":
        (
            "Phase 2 must load this frozen model. "
            "Do not substitute an experimental Phase 1 checkpoint."
        )
}


with open(
    MANIFEST_PATH,
    "w"
) as f:

    json.dump(
        final_manifest,
        f,
        indent=2
    )


print("Manifest saved:")
print(MANIFEST_PATH)

Manifest saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1F_final_model/configs/final_segmentation_manifest.json


In [ ]:
PHASE2_POINTER_PATH = (
    P1F_CONFIG_DIR /
    "phase2_segmentation_model_pointer.json"
)


phase2_pointer = {

    "model_path":
        str(
            FINAL_MODEL_PATH
        ),

    "config_path":
        str(
            FINAL_CONFIG_PATH
        ),

    "manifest_path":
        str(
            MANIFEST_PATH
        ),

    "sha256":
        final_sha256,

    "resolution":
        256,

    "threshold":
        0.50,

    "postprocessing":
        [
            "largest connected component",
            "binary hole filling"
        ],

    "frozen":
        True
}


with open(
    PHASE2_POINTER_PATH,
    "w"
) as f:

    json.dump(
        phase2_pointer,
        f,
        indent=2
    )


print("=" * 70)
print("PHASE 2 MODEL POINTER CREATED")
print("=" * 70)

print("\nPhase 2 should read:")
print(PHASE2_POINTER_PATH)

print("\nWhich points to:")
print(FINAL_MODEL_PATH)

PHASE 2 MODEL POINTER CREATED

Phase 2 should read:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1F_final_model/configs/phase2_segmentation_model_pointer.json

Which points to:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1F_final_model/models/final_lv_segmentation_model.keras


In [ ]:
required_files = {

    "Frozen model":
        FINAL_MODEL_PATH,

    "Final configuration":
        FINAL_CONFIG_PATH,

    "Provenance manifest":
        MANIFEST_PATH,

    "Phase 2 pointer":
        PHASE2_POINTER_PATH,

    "Phase 1E record":
        P1E_RECORD_PATH
}


print("=" * 70)
print("FINAL ARTIFACT CHECK")
print("=" * 70)


all_present = True


for name, path in required_files.items():

    exists = path.exists()

    print(
        f"{name:24s}: "
        f"{'✓' if exists else '✗'}"
    )

    if not exists:
        all_present = False


assert all_present


# Recheck final hash

current_final_hash = sha256_file(
    FINAL_MODEL_PATH
)

assert (
    current_final_hash
    ==
    final_sha256
)


print("\n" + "=" * 70)
print("PHASE 1F COMPLETED")
print("=" * 70)

print(
    "\nFINAL MODEL:"
)

print(FINAL_MODEL_PATH)

print(
    "\nPARAMETERS:"
)

print("7,759,521")

print(
    "\nSHA-256:"
)

print(final_sha256)

print(
    "\nFINAL CONFIGURATION:"
)

print(
    "256×256 → Large U-Net → "
    "Adam (1e-4) → batch 2 → "
    "no augmentation → BCE+Dice"
)

print(
    "\nINFERENCE:"
)

print(
    "Threshold 0.50 → "
    "largest connected component → "
    "hole filling"
)

print(
    "\nMODEL STATUS: FROZEN"
)

print(
    "POST-TEST TUNING: NONE"
)

print(
    "\n✓ PHASE 1 SEGMENTATION IS COMPLETE."
)

print(
    "\nNEXT: Phase 2 — "
    "Quantitative LV Analysis"
)

FINAL ARTIFACT CHECK
Frozen model            : ✓
Final configuration     : ✓
Provenance manifest     : ✓
Phase 2 pointer         : ✓
Phase 1E record         : ✓

PHASE 1F COMPLETED

FINAL MODEL:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1F_final_model/models/final_lv_segmentation_model.keras

PARAMETERS:
7,759,521

SHA-256:
65ec53959831c6b7c08747a5d94de6563580ce9ec052aa7b9afa614bee5705d3

FINAL CONFIGURATION:
256×256 → Large U-Net → Adam (1e-4) → batch 2 → no augmentation → BCE+Dice

INFERENCE:
Threshold 0.50 → largest connected component → hole filling

MODEL STATUS: FROZEN
POST-TEST TUNING: NONE

✓ PHASE 1 SEGMENTATION IS COMPLETE.

NEXT: Phase 2 — Quantitative LV Analysis
